# Fine-tuning con LoRA de Pixtral 12B — Entrada: imagen + texto

## Importar librerías

In [1]:
import matplotlib.pyplot as plt
from torch import nn
import pandas as pd
import numpy as np
import torch
import json
import os
import gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    roc_curve,
    auc
)

from transformers import (
    LlavaForConditionalGeneration, 
    AutoProcessor,
    TrainingArguments, 
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset as TorchDataset

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.metrics.metricfactory import MetricFactory

## Configuración y parámetros

In [2]:
os.environ["HF_TOKEN"] = ""

# ====================
# CONFIGURACIÓN MODELO
# ====================
MODEL_NAME = "mistral-community/pixtral-12b"
MAIN_PATH  = ".."
GROUP_ID   = "BeingChillingWeWillWin"
MODEL_ID   = "pixtral_12b_ft"

TEXT_COLUMN  = "combined_text"
LABEL_COLUMN = "label"

# ====================
# RUTAS DE DATOS
# ====================
DATA_TRAIN_PATH = os.path.join(MAIN_PATH, "preprocessed_data", "train_split.json")
DATA_VAL_PATH   = os.path.join(MAIN_PATH, "preprocessed_data", "dev_split.json")
DATA_TEST_PATH  = os.path.join(MAIN_PATH, "preprocessed_data", "test_split.json")

DATA_BASE_DIR   = os.path.join(MAIN_PATH, "materials", "dataset_task2_exist2026")
OUTPUT_DIR      = os.path.join(MAIN_PATH, "weights", f"Pixtral-12B_{TEXT_COLUMN}_lora")
SAVE_PATH       = os.path.join(MAIN_PATH, "weights", f"Pixtral-12B_{TEXT_COLUMN}_final")
PREDICTIONS_DIR = os.path.join(MAIN_PATH, "predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# ====================
# HIPERPARÁMETROS
# ====================
# IMPORTANTE: Aumentado para acomodar tokens de imagen de Pixtral
MAX_LENGTH     = 8192  # Pixtral puede generar 3000+ tokens por imagen 
MAX_NEW_TOKENS = 10
NUM_EPOCHS     = 3
LEARNING_RATE  = 2e-5
BATCH_SIZE     = 1
GRAD_ACCUM     = 16

# Resolución máxima de imagen para controlar tokens de imagen
# Pixtral genera tokens proporcionales a la resolución
MAX_IMAGE_SIZE = 512  # Reducir imágenes grandes para evitar token overflow

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

label_map         = {"NO": 0, "YES": 1}
label_map_inverse = {0: "NO", 1: "YES"}

## Carga y preprocesamiento de datos

In [3]:
def load_json_dataset(path):
    """Carga el JSON orientado a diccionario y devuelve un DataFrame."""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data.values())

def build_combined_text(row):
    """Construye la cadena combinada a partir de image_description y text."""
    img_desc = str(row.get('image_description', '') or '')
    txt      = str(row.get('text', '') or '')
    return f"descripcion imagen: {img_desc}. Texto: {txt}"

train_df = load_json_dataset(DATA_TRAIN_PATH)
val_df   = load_json_dataset(DATA_VAL_PATH)
test_df  = load_json_dataset(DATA_TEST_PATH)

for df in [train_df, val_df, test_df]:
    df[TEXT_COLUMN] = df.apply(build_combined_text, axis=1)

train_df["label_int"] = train_df[LABEL_COLUMN].map(label_map)
val_df["label_int"]   = val_df[LABEL_COLUMN].map(label_map)
test_df["label_int"]  = -1  # placeholder para test sin gold

print(f"Text column used : {TEXT_COLUMN}")
print(f"Ejemplo de entrada:\n  {train_df[TEXT_COLUMN].iloc[0][:200]}")
print(f"\nTrain size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"\nDistribución de etiquetas en TRAIN:")
print(train_df[LABEL_COLUMN].value_counts())
print(f"\nDistribución de etiquetas en VAL:")
print(val_df[LABEL_COLUMN].value_counts())

Text column used : combined_text
Ejemplo de entrada:
  descripcion imagen: a close up of a snake with its mouth open and its tongue out. Texto: Demostración de que las cosas mas peligrosas del mundo tienen el mismo aspecto. mémenoides 

Train size: 2146 | Val size: 537 | Test size: 687

Distribución de etiquetas en TRAIN:
label
YES    1282
NO      864
Name: count, dtype: int64

Distribución de etiquetas en VAL:
label
YES    321
NO     216
Name: count, dtype: int64


## Carga del modelo con cuantización y configuración LoRA

In [4]:
# Limpiar memoria antes de cargar
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Configuración de cuantización 8-bit para reducir uso de memoria
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=DTYPE,
)

print(f"Cargando modelo {MODEL_NAME}...")

# Cargar modelo con cuantización
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Cargar processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Configurar pad_token
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# Preparar modelo para entrenamiento con cuantización
model = prepare_model_for_kbit_training(model)

# Configuración LoRA - apuntamos a los módulos de atención del modelo
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Módulos de atención
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Aplicar LoRA al modelo
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n✓ Modelo Pixtral 12B cargado con LoRA")
if torch.cuda.is_available():
    print(f"  Memoria GPU usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Cargando modelo mistral-community/pixtral-12b...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

The image processor of type `PixtralImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


trainable params: 22,806,528 || all params: 12,705,546,240 || trainable%: 0.1795

✓ Modelo Pixtral 12B cargado con LoRA
  Memoria GPU usada: 3.41 GB


## Dataset y Data Collator para Fine-tuning

In [5]:
def resize_image_if_needed(image, max_size=MAX_IMAGE_SIZE):
    """
    Redimensiona imagen si excede max_size manteniendo aspect ratio.
    Esto es CRÍTICO para Pixtral porque el número de tokens de imagen
    es proporcional a la resolución.
    """
    width, height = image.size
    if max(width, height) > max_size:
        if width > height:
            new_width = max_size
            new_height = int(height * (max_size / width))
        else:
            new_height = max_size
            new_width = int(width * (max_size / height))
        image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)
    return image


class SexismMemeDataset(TorchDataset):
    """Dataset para clasificación de memes con imagen + texto para Pixtral."""
    
    def __init__(self, df, processor, base_dir, max_length=8192, max_image_size=512):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.base_dir = Path(base_dir)
        self.max_length = max_length
        self.max_image_size = max_image_size
        # Lista de índices válidos (se llena después de validación)
        self.valid_indices = list(range(len(self.df)))
        
    def __len__(self):
        return len(self.valid_indices)
    
    def _load_and_resize_image(self, img_path):
        """Carga y redimensiona imagen."""
        try:
            image = Image.open(img_path).convert('RGB')
            image = resize_image_if_needed(image, self.max_image_size)
            return image
        except Exception as e:
            print(f"[WARN] Error cargando imagen {img_path}: {e}")
            return Image.new('RGB', (224, 224), color='gray')
    
    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        
        # Cargar y redimensionar imagen
        img_path = self.base_dir / row['path_memes']
        image = self._load_and_resize_image(img_path)
        
        # Construir prompt para Pixtral
        combined_text = row[TEXT_COLUMN]
        label = row['label_int']
        target_text = "YES" if label == 1 else "NO"
        
        system_instruction = (
            "You are an expert content moderator. Analyze the meme and determine "
            "if it contains sexist content. Answer only YES or NO."
        )
        user_content = f"[IMG]\\nAnalyze this meme. {combined_text}\\n\\nIs this meme sexist? Answer YES or NO."
        prompt = f"[INST]{system_instruction}\\n\\n{user_content}[/INST]{target_text}"
        
        # IMPORTANTE: No usar truncation=True con Pixtral - causa el error de token mismatch
        # En su lugar, confiamos en que las imágenes redimensionadas caben en max_length
        encoding = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding=False,  # El collator hace el padding
        )
            
        item = {}
        for k, v in encoding.items():
            if k in ['pixel_values', 'image_sizes']:
                item[k] = v
            elif isinstance(v, torch.Tensor):
                item[k] = v.squeeze(0)
            else:
                item[k] = v
        
        # Truncar manualmente si es necesario (DESPUÉS del encoding)
        if item['input_ids'].shape[0] > self.max_length:
            item['input_ids'] = item['input_ids'][:self.max_length]
            item['attention_mask'] = item['attention_mask'][:self.max_length]
        
        item['labels'] = item['input_ids'].clone()
        
        return item


def validate_dataset_fast(dataset, processor, max_samples=None, batch_size=50):
    """
    Valida el dataset ANTES del entrenamiento para detectar errores rápidamente.
    Devuelve lista de índices válidos y lista de índices problemáticos.
    """
    print("=" * 60)
    print("VALIDACIÓN RÁPIDA DEL DATASET (fail-fast)")
    print("=" * 60)
    
    n_samples = len(dataset.df)
    if max_samples:
        n_samples = min(n_samples, max_samples)
    
    valid_indices = []
    failed_indices = []
    failed_reasons = {}
    
    # Procesar en fragmentos para reportar progreso frecuente
    for batch_start in tqdm(range(0, n_samples, batch_size), desc="Validando fragmentos"):
        batch_end = min(batch_start + batch_size, n_samples)
        
        for idx in range(batch_start, batch_end):
            row = dataset.df.iloc[idx]
            img_path = dataset.base_dir / row['path_memes']
            
            try:
                # Cargar y redimensionar imagen
                image = dataset._load_and_resize_image(img_path)
                
                # Construir prompt
                combined_text = row[TEXT_COLUMN]
                system_instruction = (
                    "You are an expert content moderator. Analyze the meme and determine "
                    "if it contains sexist content. Answer only YES or NO."
                )
                user_content = f"[IMG]\\nAnalyze this meme. {combined_text}\\n\\nIs this meme sexist? Answer YES or NO."
                prompt = f"[INST]{system_instruction}\\n\\n{user_content}[/INST]YES"
                
                # Intentar procesar (sin truncación para detectar problemas)
                encoding = processor(
                    text=prompt,
                    images=image,
                    return_tensors="pt",
                    padding=False,
                )
                
                # Verificar longitud
                seq_len = encoding['input_ids'].shape[1]
                if seq_len > dataset.max_length:
                    # Advertencia pero aún válido (se truncará después)
                    print(f"[WARN] idx {idx}: seq_len={seq_len} > max_length={dataset.max_length}, se truncará")
                
                valid_indices.append(idx)
                
            except Exception as e:
                failed_indices.append(idx)
                failed_reasons[idx] = str(e)
        
        # Reporte intermedio cada fragmento
        if len(failed_indices) > 0 and batch_start % (batch_size * 5) == 0:
            print(f"  [Fragmento {batch_start//batch_size + 1}] Válidos: {len(valid_indices)}, Fallidos: {len(failed_indices)}")
    
    print(f"\n{'=' * 60}")
    print(f"RESULTADO VALIDACIÓN:")
    print(f"  Total muestras: {n_samples}")
    print(f"  Válidas: {len(valid_indices)}")
    print(f"  Fallidas: {len(failed_indices)}")
    
    if failed_indices:
        print(f"\nMuestras fallidas (primeras 10):")
        for idx in failed_indices[:10]:
            print(f"  idx {idx}: {failed_reasons[idx][:100]}...")
    
    print("=" * 60)
    
    return valid_indices, failed_indices, failed_reasons


class DataCollatorForVLM:
    """Collator para Vision-Language Model (Pixtral)."""
    
    def __init__(self, processor, pad_token_id=None):
        self.processor = processor
        self.pad_token_id = pad_token_id or processor.tokenizer.pad_token_id
        
    def __call__(self, features):
        batch = {}
        
        # Handle input_ids - pad a la longitud máxima del batch
        if 'input_ids' in features[0]:
            max_length = max(f['input_ids'].shape[0] for f in features)
            input_ids_list = []
            for f in features:
                ids = f['input_ids']
                padding_length = max_length - ids.shape[0]
                if padding_length > 0:
                    ids = torch.cat([ids, torch.full((padding_length,), self.pad_token_id, dtype=ids.dtype)])
                input_ids_list.append(ids)
            batch['input_ids'] = torch.stack(input_ids_list)
            
        # Handle attention_mask - pad con 0s
        if 'attention_mask' in features[0]:
            max_length = max(f['attention_mask'].shape[0] for f in features)
            attention_mask_list = []
            for f in features:
                mask = f['attention_mask']
                padding_length = max_length - mask.shape[0]
                if padding_length > 0:
                    mask = torch.cat([mask, torch.zeros(padding_length, dtype=mask.dtype)])
                attention_mask_list.append(mask)
            batch['attention_mask'] = torch.stack(attention_mask_list)
            
        # Handle pixel_values
        if 'pixel_values' in features[0]:
            pixel_values_list = [f['pixel_values'] for f in features]
            batch['pixel_values'] = torch.cat(pixel_values_list, dim=0)
        
        # Handle image_sizes
        if 'image_sizes' in features[0]:
            image_sizes_list = [f['image_sizes'] for f in features]
            if isinstance(image_sizes_list[0], torch.Tensor):
                batch['image_sizes'] = torch.cat(image_sizes_list, dim=0)
            else:
                batch['image_sizes'] = image_sizes_list
            
        # Handle labels - pad y marcar pad tokens como -100
        if 'labels' in features[0]:
            max_length = max(f['labels'].shape[0] for f in features)
            labels_list = []
            for f in features:
                labs = f['labels']
                padding_length = max_length - labs.shape[0]
                if padding_length > 0:
                    labs = torch.cat([labs, torch.full((padding_length,), -100, dtype=labs.dtype)])
                labels_list.append(labs)
            batch['labels'] = torch.stack(labels_list)
            batch['labels'][batch['labels'] == self.pad_token_id] = -100
            
        return batch


# Crear datasets
print("Creando datasets...")
train_dataset = SexismMemeDataset(train_df, processor, DATA_BASE_DIR, MAX_LENGTH, MAX_IMAGE_SIZE)
eval_dataset = SexismMemeDataset(val_df, processor, DATA_BASE_DIR, MAX_LENGTH, MAX_IMAGE_SIZE)

# VALIDACIÓN RÁPIDA - Detecta errores ANTES del entrenamiento largo
print("\n>>> VALIDANDO TRAIN DATASET...")
valid_train, failed_train, reasons_train = validate_dataset_fast(train_dataset, processor)
train_dataset.valid_indices = valid_train

print("\n>>> VALIDANDO EVAL DATASET...")
valid_eval, failed_eval, reasons_eval = validate_dataset_fast(eval_dataset, processor)
eval_dataset.valid_indices = valid_eval

if len(failed_train) > 0 or len(failed_eval) > 0:
    print("\n" + "!"*60)
    print("ADVERTENCIA: Algunas muestras fallaron la validación.")
    print("Se excluirán del entrenamiento para evitar errores.")
    print("!"*60)

print(f"\n✓ Train dataset FINAL: {len(train_dataset)} ejemplos (excluidos {len(failed_train)})")
print(f"✓ Eval dataset FINAL: {len(eval_dataset)} ejemplos (excluidos {len(failed_eval)})")

# Verificar un ejemplo
sample = train_dataset[0]
print(f"\nEjemplo de entrada:")
print(f"  input_ids shape: {sample['input_ids'].shape}")
if 'pixel_values' in sample:
    print(f"  pixel_values shape: {sample['pixel_values'].shape}")
print(f"  labels shape: {sample['labels'].shape}")

Creando datasets...

>>> VALIDANDO TRAIN DATASET...
VALIDACIÓN RÁPIDA DEL DATASET (fail-fast)


Validando fragmentos:   0%|          | 0/43 [00:00<?, ?it/s]


RESULTADO VALIDACIÓN:
  Total muestras: 2146
  Válidas: 2146
  Fallidas: 0

>>> VALIDANDO EVAL DATASET...
VALIDACIÓN RÁPIDA DEL DATASET (fail-fast)


Validando fragmentos:   0%|          | 0/11 [00:00<?, ?it/s]


RESULTADO VALIDACIÓN:
  Total muestras: 537
  Válidas: 537
  Fallidas: 0

✓ Train dataset FINAL: 2146 ejemplos (excluidos 0)
✓ Eval dataset FINAL: 537 ejemplos (excluidos 0)

Ejemplo de entrada:
  input_ids shape: torch.Size([962])
  pixel_values shape: torch.Size([1, 3, 512, 416])
  labels shape: torch.Size([962])


## Configuración del Trainer

In [6]:
# Data collator
data_collator = DataCollatorForVLM(processor)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,  # Importante para mantener pixel_values
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Para mejor compatibilidad
    optim="paged_adamw_8bit",  # Optimizador que ahorra memoria
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer configurado")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Trainer configurado


## Entrenamiento

In [ ]:
# Ejecutar entrenamiento
print("Iniciando entrenamiento...")
trainer.train()

# Guardar modelo final
print("\nGuardando modelo...")
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"✓ Modelo guardado en: {SAVE_PATH}")

Iniciando entrenamiento...


/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss


## Inferencia en DEV y cálculo de métricas

In [ ]:
@torch.no_grad()
def generate_prediction(row, model, processor, base_dir, max_image_size=MAX_IMAGE_SIZE):
    """Genera predicción para una muestra con Pixtral."""
    img_path = Path(base_dir) / row['path_memes']
    combined_text = row[TEXT_COLUMN]
    
    try:
        image = Image.open(img_path).convert('RGB')
        # IMPORTANTE: Redimensionar igual que en entrenamiento
        image = resize_image_if_needed(image, max_image_size)
    except:
        return {'classification': 'NO', 'confidence': 0.0}
    
    # Construir prompt con [IMG] token para Pixtral - debe coincidir con entrenamiento
    system_instruction = (
        "You are an expert content moderator. Analyze the meme and determine "
        "if it contains sexist content. Answer only YES or NO."
    )
    user_content = f"[IMG]\\nAnalyze this meme. {combined_text}\\n\\nIs this meme sexist? Answer YES or NO."
    prompt = f"[INST]{system_instruction}\\n\\n{user_content}[/INST]"
    
    # Sin truncation - las imágenes ya están redimensionadas
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=MAX_NEW_TOKENS, 
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id
    )
    response = processor.decode(outputs[0], skip_special_tokens=True)
    response = response.split("[/INST]")[-1].strip().upper() if "[/INST]" in response else response.upper()
    
    if "YES" in response:
        return {'classification': 'YES', 'confidence': 0.9}
    else:
        return {'classification': 'NO', 'confidence': 0.9}


def save_probs_json(ids, probs, split_name, labels=None):
    records = []
    for i, (id_exist, prob) in enumerate(zip(ids, probs)):
        rec = {'id': str(id_exist), 'prob_YES': round(float(prob), 6)}
        if labels is not None:
            rec['label'] = label_map_inverse[int(labels[i])]
        records.append(rec)
    path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_{split_name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def evaluate_in_chunks(df, model, processor, base_dir, chunk_size=100, desc="Inferencia"):
    """
    Evalúa el dataset en fragmentos para detectar errores rápidamente.
    Guarda resultados parciales después de cada fragmento.
    """
    all_results = []
    n_chunks = (len(df) + chunk_size - 1) // chunk_size
    
    for chunk_idx in range(n_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, len(df))
        chunk_df = df.iloc[start_idx:end_idx]
        
        chunk_results = []
        for _, row in tqdm(chunk_df.iterrows(), 
                          total=len(chunk_df), 
                          desc=f"{desc} [Chunk {chunk_idx+1}/{n_chunks}]"):
            try:
                pred = generate_prediction(row, model, processor, base_dir)
                chunk_results.append({
                    'id_EXIST': str(row['id_EXIST']),
                    'classification': pred['classification'],
                    'confidence': pred['confidence'],
                    'true_label': label_map_inverse.get(row.get('label_int', -1), 'UNKNOWN')
                })
            except Exception as e:
                print(f"[ERROR] Fallo en id {row.get('id_EXIST', '?')}: {e}")
                chunk_results.append({
                    'id_EXIST': str(row['id_EXIST']),
                    'classification': 'NO',  # Default fallback
                    'confidence': 0.0,
                    'true_label': label_map_inverse.get(row.get('label_int', -1), 'UNKNOWN'),
                    'error': str(e)
                })
        
        all_results.extend(chunk_results)
        
        # Guardar checkpoint después de cada chunk
        checkpoint_path = os.path.join(PREDICTIONS_DIR, f'checkpoint_{desc.lower().replace(" ", "_")}_chunk{chunk_idx+1}.json')
        with open(checkpoint_path, 'w') as f:
            json.dump(all_results, f, indent=2)
        print(f"  Checkpoint guardado: {checkpoint_path}")
    
    return all_results


# Evaluar en DEV con chunks
model.eval()
print("Evaluando en DEV (en fragmentos)...")

val_results = evaluate_in_chunks(val_df, model, processor, DATA_BASE_DIR, chunk_size=100, desc="DEV")

# Filtrar errores para métricas
val_results_ok = [r for r in val_results if 'error' not in r]
print(f"\nResultados válidos: {len(val_results_ok)}/{len(val_results)}")

y_pred_labels = [label_map[r['classification']] for r in val_results_ok]
y_true_labels = [label_map[r['true_label']] for r in val_results_ok]
y_probs_dev = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in val_results_ok]

# Métricas
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_labels, y_pred_labels, average='binary', zero_division=0
)

# Guardar IDs válidos para el JSON de probs
valid_ids = [r['id_EXIST'] for r in val_results_ok]
valid_labels = [label_map[r['true_label']] for r in val_results_ok]
save_probs_json(valid_ids, y_probs_dev, 'dev', labels=valid_labels)

print(f"\n=== Métricas en DEV (Fine-tuned) ===")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall   : {recall:.4f}")
print(f"  F1-Score : {f1:.4f}")

# ROC AUC
fpr, tpr, _ = roc_curve(y_true_labels, y_probs_dev)
roc_auc = auc(fpr, tpr)
print(f"\nAUC (DEV): {roc_auc:.4f}")

# Gráfica ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve DEV — Pixtral 12B Fine-tuned')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluación en DEV con PyEvALL

In [ ]:
# Preparar datos para PyEvALL (solo resultados válidos)
dev_preds_for_pyevall = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in val_results_ok
]
dev_preds_df = pd.DataFrame(dev_preds_for_pyevall)
dev_preds_path = os.path.join(PREDICTIONS_DIR, 'dev_predictions_temp.json')
with open(dev_preds_path, 'w', encoding='utf-8') as f:
    f.write(dev_preds_df.to_json(orient='records'))

# Gold labels - solo para IDs válidos
valid_ids_set = set(r['id_EXIST'] for r in val_results_ok)
dev_gold = [
    {'test_case': 'EXIST2025', 'id': str(id_exist), 'value': label}
    for id_exist, label in zip(val_df['id_EXIST'].values, val_df[LABEL_COLUMN].values)
    if str(id_exist) in valid_ids_set
]
dev_gold_df = pd.DataFrame(dev_gold)
dev_gold_path = os.path.join(PREDICTIONS_DIR, 'dev_gold_temp.json')
with open(dev_gold_path, 'w', encoding='utf-8') as f:
    f.write(dev_gold_df.to_json(orient='records'))

# Evaluar con PyEvALL
test_eval = PyEvALLEvaluation()
metrics = [MetricFactory.Accuracy.value, MetricFactory.FMeasure.value]
report = test_eval.evaluate(dev_preds_path, dev_gold_path, metrics)
print("\n=== Evaluación en DEV con PyEvALL ===")
report.print_report()

## Inferencia en TEST y generación de predicciones finales

In [ ]:
# Inferencia en TEST con chunks
print("Evaluando en TEST (en fragmentos)...")

test_results = evaluate_in_chunks(test_df, model, processor, DATA_BASE_DIR, chunk_size=100, desc="TEST")

# Filtrar errores si los hubiera
test_results_ok = [r for r in test_results if 'error' not in r]
print(f"\nResultados válidos: {len(test_results_ok)}/{len(test_results)}")

y_probs_test = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in test_results_ok]
test_preds = [r['classification'] for r in test_results_ok]

# Guardar probs solo para resultados válidos
valid_test_ids = [r['id_EXIST'] for r in test_results_ok]
save_probs_json(valid_test_ids, y_probs_test, 'test')

print(f"\nPredicciones en TEST:")
print(f"  Total: {len(test_preds)}")
print(f"  YES  : {sum(1 for p in test_preds if p == 'YES')} ({100*sum(1 for p in test_preds if p == 'YES')/len(test_preds):.2f}%)")
print(f"  NO   : {sum(1 for p in test_preds if p == 'NO')} ({100*sum(1 for p in test_preds if p == 'NO')/len(test_preds):.2f}%)")

## Guardar predicciones en formato PyEvALL para TEST

In [ ]:
# Guardar predicciones en formato PyEvALL
# Usar test_results_ok para excluir errores
test_preds_for_submission = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in test_results_ok
]
test_preds_df = pd.DataFrame(test_preds_for_submission)

output_filename = f"{GROUP_ID}_{MODEL_ID}.json"
output_path = os.path.join(PREDICTIONS_DIR, output_filename)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(test_preds_df.to_json(orient='records'))

print(f"\n✓ Predicciones guardadas en: {output_path}")
print(f"  Total muestras: {len(test_preds_for_submission)}")

# Guardar también lista de errores si los hay
if len(test_results) > len(test_results_ok):
    errors_path = os.path.join(PREDICTIONS_DIR, f"{GROUP_ID}_{MODEL_ID}_errors.json")
    test_errors = [r for r in test_results if 'error' in r]
    with open(errors_path, 'w') as f:
        json.dump(test_errors, f, indent=2)
    print(f"⚠ {len(test_errors)} errores guardados en: {errors_path}")